<a href="https://colab.research.google.com/github/husn44/bnpl-default-prediction/blob/husna-branch/notebooks/01_data_generation_A.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

BNPL Default Prediction - Data Generation (Member A)

In [1]:
# ---------------------------------------------------------------------
# STEP 1 — Setup + demographics
# ---------------------------------------------------------------------
import os
import numpy as np
import pandas as pd

N = 3000                 # number of customers (assignment minimum is 1000)
SEED = 42
TARGET_DEFAULT_RATE = 0.20
rng = np.random.default_rng(SEED)

customer_id = [f"C{i:05d}" for i in range(1, N + 1)]

# BNPL users skew young
age = np.clip(rng.normal(29, 8, N), 18, 60).round().astype(int)

gender = rng.choice(["Male", "Female"], N, p=[0.48, 0.52])

states = ["Selangor", "Kuala Lumpur", "Johor", "Penang", "Perak",
          "Sabah", "Sarawak", "Kedah", "Negeri Sembilan", "Melaka"]
state_p = [0.24, 0.16, 0.12, 0.09, 0.08, 0.08, 0.08, 0.06, 0.05, 0.04]
state = rng.choice(states, N, p=state_p)

# Employment type depends on age (students are young)
emp_types = ["Full-time", "Part-time", "Gig worker", "Student", "Self-employed"]
employment_type = np.empty(N, dtype=object)
for i, a in enumerate(age):
    if a <= 23:
        p = [0.25, 0.20, 0.15, 0.35, 0.05]
    elif a <= 35:
        p = [0.55, 0.10, 0.18, 0.05, 0.12]
    else:
        p = [0.60, 0.08, 0.12, 0.00, 0.20]
    employment_type[i] = rng.choice(emp_types, p=p)

# Monthly income (RM): log-normal, median depends on employment type
income_median = {"Full-time": 4500, "Part-time": 1800, "Gig worker": 2600,
                 "Student": 900, "Self-employed": 4200}
med = np.array([income_median[e] for e in employment_type])
age_boost = 1 + np.clip(age - 25, 0, 20) * 0.02          # older -> earn a bit more
monthly_income = np.clip(med * age_boost * rng.lognormal(0, 0.45, N), 600, 25000).round(2)
